### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="garments_worker_productivity",
    dataset_year="2020",
    domain_str="industry & manufacturing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C51S6D",
    download_description=r"""
mkdir -p local-data-warehouse/garments_worker_productivity \
&& wget -P local-data-warehouse/garments_worker_productivity/ https://archive.ics.uci.edu/static/public/597/productivity+prediction+of+garment+employees.zip \
&& unzip local-data-warehouse/garments_worker_productivity/productivity+prediction+of+garment+employees.zip -d local-data-warehouse/garments_worker_productivity/ \
&&  rm local-data-warehouse/garments_worker_productivity/productivity+prediction+of+garment+employees.zip
""",
    # References
    academic_reference_bibtex="""@article{imran2021mining,
    title={Mining the productivity data of the garment industry},
    author={Imran, Abdullah Al and Rahim, Md Shamsur and Ahmed, Tanvir},
    journal={International Journal of Business Intelligence and Data Mining},
    volume={19},
    number={3},
    pages={319--342},
    year={2021},
    publisher={Inderscience Publishers (IEL)}
}
""",
    academic_reference_bibtex_key="imran2021mining",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Temporal"],
    curation_comments="""
- We fix typos in data entries (e.g., "finishing " becomes "finishing").
- We transform the date to datetime.
- The associated paper conceptualizes the task as an interpretable ML task without consideration of realistic predictive scenarios. Therefore, we define a new predictive ML task.
- We define the decision point in time as the start of the day.
- We set the target variable to "actual_productivity", and use the targeted_productivity as an input feature.
- We do not include all features directly for prediction, because some can't be expected to be available at the decision point in time: "smv", "wip", "over_time", "idle_time", "idle_men". To still keep as much information as possible, we impute the value of the previous recorded working day (by department and team).
- Because the timestamps are irregularly spaced, we additionally include the days passed sind the last recording per department and team.
- Note that we try to keep feature engineering minimalistic and only with the sole purpose to prevent leaks with minimal loss of information.
- It can be expected that feature engineering exposing the non-iid properties of the dataset will be crucial for good predictive performance.
- We cannot fully exclude the possibility of leaks, since not enough information about the date of collection for some features is given.
- We define 30 splits with one day for testing each, resulting in small test sizes of 17-24 samples per split.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="actual_productivity",
    problem_type="regression",
    objective_metric_name="rmse",
    time_on="date"
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

# NOTE: "date", "quarter", "department", "day", "team", "targeted_productivity", "actual_productivity" are safe to keep. For "incentive", "no_of_style_change", "no_of_workers" it's a guess.
lag_cols = ["smv", "wip", "over_time", "idle_time", "idle_men"]

df = pd.read_csv(dataset_mold.path / "garments_worker_productivity.csv")

df.department = df.department.str.strip(" ")
df.date = pd.to_datetime(df.date)

entity_cols = ["department", "team"]
date_col = "date"
lags = (1,)

assert not df.duplicated(["department", "team", "date"]).any(), \
    "Found duplicate rows for the same department-team-date."

df = df.sort_values(list(entity_cols) + [date_col]).reset_index(drop=True)

g = df.groupby(list(entity_cols), sort=False)

prev_date = g[date_col].shift(1)
df["days_since_prev_obs"] = (df[date_col] - prev_date).dt.days

for col in lag_cols:
    for lag in lags:
        df[f"{col}_lag_{lag}"] = g[col].shift(lag)

df = df.drop(columns=lag_cols)

as_cat_type = ["quarter", "department", "day", "team"]
df[as_cat_type] = df[as_cat_type].astype("category")

print("Loaded data shape:", df.shape)
df = df.sort_values(by=date_col).reset_index(drop=True)

Loaded data shape: (1197, 16)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,197
Columns: 16
Use sampling: False (sample size: 1,197)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['wip_lag_1', 'over_time_lag_1', 'smv_lag_1', 'no_of_workers', 'date', 'incentive', 'days_since_prev_obs', 'idle_time_lag_1', 'team', 'idle_men_lag_1']
Rows remaining as candidates after top-10 filter: 0 (of 1,197)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,date,quarter,department,day,team,targeted_productivity,incentive,no_of_style_change,no_of_workers,actual_productivity,days_since_prev_obs,smv_lag_1,wip_lag_1,over_time_lag_1,idle_time_lag_1,idle_men_lag_1
0,2015-01-01,Quarter1,finishing,Thursday,1,0.75,0,0,8.0,0.886500,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-01,Quarter1,sweing,Thursday,12,0.80,50,0,30.5,0.800570,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-01,Quarter1,finishing,Thursday,4,0.75,0,0,18.0,0.593056,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-01-01,Quarter1,sweing,Thursday,11,0.80,50,0,30.5,0.800570,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-01-01,Quarter1,finishing,Thursday,7,0.80,0,0,8.0,0.540729,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,quarter,category,0.0,0.00,5.0,"Quarter1, Quarter2, Quarter4, Quarter3, Quarter5"
1,department,category,0.0,0.00,2.0,"sweing, finishing"
2,day,category,0.0,0.00,6.0,"Wednesday, Sunday, Tuesday, Monday, Thursday, Saturday"
3,team,category,0.0,0.00,12.0,"2, 8, 4, 1, 9, 10, 12, 7, 3, 6"
4,date,datetime64[ns],0.0,0.00,59.0,"2015-01-31 00:00:00, 2015-03-11 00:00:00, 2015-01-11 00:00:00, 2015-01-24 00:00:00, 2015-01-12 00:00:00, 2015-03-10 00:00:00, 2015-01-08 00:00:00, 2015-01-13 00:00:00, 2015-01-22 00:00:00, 2015-01-10 00:00:00"
5,wip_lag_1,float64,518.0,43.27,540.0,"1039.0, 1282.0, 759.0, 970.0, 1108.0, 968.0, 1079.0, 1193.0, 1069.0, 1263.0"
6,days_since_prev_obs,float64,24.0,2.01,14.0,"1.0, 2.0, 3.0, 4.0, 5.0, 7.0, 6.0, 8.0, 11.0, 15.0"
7,smv_lag_1,float64,24.0,2.01,70.0,"3.94, 2.9, 22.52, 30.1, 4.15, 18.79, 4.6, 15.26, 25.9, 11.61"
8,over_time_lag_1,float64,24.0,2.01,143.0,"960.0, 1440.0, 6960.0, 6840.0, 1200.0, 1800.0, 10170.0, 0.0, 3360.0, 4080.0"
9,idle_time_lag_1,float64,24.0,2.01,12.0,"0.0, 3.5, 2.0, 8.0, 4.0, 4.5, 5.0, 150.0, 300.0, 90.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
targeted_productivity,1197.0,0.729632,0.097891,0.070000,0.800000
incentive,1197.0,38.210526,160.182643,0.000000,3600.000000
no_of_style_change,1197.0,0.150376,0.427848,0.000000,2.000000
no_of_workers,1197.0,34.609858,22.197687,2.000000,89.000000
actual_productivity,1197.0,0.735091,0.174488,0.233705,1.120437
days_since_prev_obs,1173.0,1.384484,1.101348,1.000000,16.000000
smv_lag_1,1173.0,15.095303,10.945454,2.900000,54.560000
wip_lag_1,679.0,1195.238586,1852.668096,7.000000,23122.000000
over_time_lag_1,1173.0,4584.748508,3364.083787,0.000000,25920.000000
idle_time_lag_1,1173.0,0.745098,12.838797,0.000000,300.000000


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column     rank                                   
date       1     2015-01-31 00:00:00     24   2.01
           2     2015-03-11 00:00:00     24   2.01
           3     2015-01-11 00:00:00     23   1.92
           4     2015-01-24 00:00:00     23   1.92
           5     2015-01-12 00:00:00     23   1.92
day        1               Wednesday    208  17.38
           2                  Sunday    203  16.96
           3                 Tuesday    201  16.79
           4                  Monday    199  16.62
           5                Thursday    199  16.62
department 1                  sweing    691  57.73
           2               finishing    506  42.27
quarter    1                Quarter1    360  30.08
           2                Quarter2    335  27.99
           3                Quarter4    248  20.72
           4                Quarter3    210  17.54
           5                Quarter5     44   3.68
team       1                       2    109   9.11
           2                       8    109   9.11
           3                       4    105   8.77
           4                       1    105   8.77
           5                       9    104   8.69

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-0.807,-1.574,0.03,0.084,log,1859.1,4.878156e+15,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata

date_col = task_mold.time_on
target_col = task_mold.target_column_name

df = df.sort_values(by=date_col).reset_index(drop=True)

used_in_train = set()
used_in_test = set()
used_data = set()

splits = {}

for split, date in enumerate(sorted(df[date_col].unique())[::-1][:30]):
    train_idx = df[df[date_col] < date].index.tolist()
    test_idx = df[df[date_col] == date].index.tolist()

    splits[split] = {0: [train_idx, test_idx]}

    used_in_train.update(train_idx)
    used_in_test.update(test_idx)
    used_data.update(train_idx)
    used_data.update(test_idx)

    print(f"\n=== Step {split} ===")
    print("Train size:", len(train_idx), "| Test size:", len(test_idx))
    print("Train target mean:", df.loc[train_idx, target_col].mean())
    print("Test target mean:", df.loc[test_idx, target_col].mean())

    assert len(set(train_idx).intersection(set(test_idx)))==0, "Train and test indices overlap!"

print(f"{len(used_data)/df.shape[0]:.4f} of the samples are used.")
print(f"{len(used_in_train)/df.shape[0]:.4f} of the samples are used in training")
print(f"{len(used_in_test)/df.shape[0]:.4f} of the samples are used in testing.")

splits_mold = PredictiveMLSplitsMetadata( 
    splits_comment=r"We define 30 splits with one day for testing each, resulting in small test sizes of 17-24 samples per split. The split with the smallest train sizes uses exactly half of the available samples.",
    splits=splits,
    # We refit every month, so we understand the horizon to be based on months.
    time_horizon=1,
    time_horizon_unit="days",
)


=== Step 0 ===
Train size: 1173 | Test size: 24
Train target mean: 0.7352367291884059
Test target mean: 0.7279733227500002

=== Step 1 ===
Train size: 1150 | Test size: 23
Train target mean: 0.7348458387686957
Test target mean: 0.7547812501739131

=== Step 2 ===
Train size: 1128 | Test size: 22
Train target mean: 0.7349636862234044
Test target mean: 0.7288034783636363

=== Step 3 ===
Train size: 1106 | Test size: 22
Train target mean: 0.7349156189294755
Test target mean: 0.7373801601818183

=== Step 4 ===
Train size: 1087 | Test size: 19
Train target mean: 0.7353278092594296
Test target mean: 0.7113339932105263

=== Step 5 ===
Train size: 1068 | Test size: 19
Train target mean: 0.73539477785206
Test target mean: 0.7315634694210525

=== Step 6 ===
Train size: 1047 | Test size: 21
Train target mean: 0.7357302658385864
Test target mean: 0.7186683053809524

=== Step 7 ===
Train size: 1025 | Test size: 22
Train target mean: 0.7371274814682927
Test target mean: 0.6706327194545455

=== Step 

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to garments_worker_productivity/019db501-f4c4-7de2-b626-fcb769c7d82e
019db501-f4c4-7de2-b626-fcb769c7d82e
afa4f62c0fdb65ec7a313670ea6bfda0acb40e9d4b84118ac3c1907ca62cab6f
